# INF201 Deliverable 3: Battle simulator

Authors: Sanji Miller and Sajjad Ahmadi

GitHub repository: https://github.com/sanji2006/Deliverable_3

A mini-RPG where two characters battle until one wins. The classes are in separate modules: `dice.py`, `items.py`, `character.py` and `battle.py`.

In [1]:
from dice import Dice
from items import Weapon
from character import Character, Warrior, Mage
from battle import Battle

## Overview of the classes

The helper below lists the public methods and properties of each class. The
attributes of the objects are shown in the sections for each class.

In [2]:
def show_class(cls: type) -> None:
    """Print the public methods and properties of a class."""
    names = [n for n in dir(cls) if not n.startswith('_') or n in ('__call__', '__repr__')]
    print(f"{cls.__name__}: {names}")


for cls in (Dice, Weapon, Character, Warrior, Mage, Battle):
    show_class(cls)

Dice: ['__call__', '__repr__', 'roll']
Weapon: ['__repr__', 'roll_damage']
Character: ['__repr__', 'attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']
Warrior: ['__repr__', 'attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']
Mage: ['__repr__', 'attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']
Battle: ['__repr__', 'play_round', 'simulate_battle']


## Dice

`Dice(sides=6)` is an n-sided dice (at least 2 sides). `roll()` returns a
random number from 1 to `sides`. `__repr__` gives `Dice(sides=6)`, and
`__call__` lets us write `d20()` instead of `d20.roll()`.

In [3]:
d20 = Dice(sides=20)
print(d20, d20.roll(), d20())
print("Attributes:", vars(d20))

Dice(sides=20) 11 12
Attributes: {'sides': 20}


## Weapon

A `Weapon` has a `name` and a `damage_dice` (a `Dice`). `roll_damage()`
rolls the dice and returns the damage.

In [4]:
sword = Weapon("Sword", Dice(8))
print(sword.name, sword.damage_dice, sword.roll_damage())
print(vars(sword))

Sword Dice(sides=8) 4
{'name': 'Sword', 'damage_dice': Dice(sides=8)}


## Character

A `Character` has `name`, `hp_max`, `damage` and an optional `weapon`. HP is
protected (`_hp`) and read through the `hp` property. `take_damage()` and
`heal()` change only the character's own HP, and `is_alive` is a read-only
property. `attack()` calls the target's `take_damage()`, using the weapon's
damage if one is equipped.

In [5]:
a = Character("Aria")
b = Character("Brom", hp_max=30)
a.attack(b)
print(b.hp, b.is_alive)   # 20 True
b.heal(100)
print(b.hp)               # 30, capped at hp_max
try:
    b.is_alive = True
except AttributeError as e:
    print("is_alive is read-only:", e)

20 True
30
is_alive is read-only: property 'is_alive' of 'Character' object has no setter


## Warrior and Mage

`Warrior` has `armor` (default 5) that reduces incoming damage in
`take_damage()`. `Mage` has `mana` and `mana_max` (default 30). With at least
10 mana a mage casts a spell for double damage (no weapon), otherwise it uses
the base attack with its weapon and restores 5 mana.

In [6]:
w = Warrior("Brom")
print(w.armor, vars(w))
w.take_damage(12)
print(w.hp)   # 93, since 12 - 5 = 7 damage

m = Mage("Aria")
target = Character("Dummy")
m.attack(target)
print(target.hp, m.mana)   # 80 20
print(vars(m))

5 {'name': 'Brom', 'hp_max': 100, '_hp': 100, 'damage': 10, 'weapon': None, 'armor': 5}
93
80 20
{'name': 'Aria', 'hp_max': 100, '_hp': 100, 'damage': 10, 'weapon': None, 'mana': 20, 'mana_max': 30}


A mage with less than 10 mana falls back to the base attack, which uses the
equipped weapon, and restores 5 mana.

In [7]:
m2 = Mage("Merlin", mana=5, weapon=Weapon("Staff", Dice(6)))
t = Character("Dummy")
m2.attack(t)
print(t.hp, m2.mana)   # damage 1-6 from the staff, mana is now 10

96 10


## Battle

`Battle(hero, enemy)` manages turn-based combat. `play_round()` lets the hero
attack, and the enemy retaliates if still alive. `simulate_battle()` repeats
rounds until one character is dead.

In [8]:
demo = Battle(Warrior("A"), Mage("B"))
print(vars(demo))

{'hero': <Character name=A hp=100 damage=10>, 'enemy': <Character name=B hp=100 damage=10>, 'round_number': 0}


In [9]:
# Without weapons: Warrior vs Mage
Battle(Warrior("Brom"), Mage("Aria")).simulate_battle()

Round 1: Brom 85 HP | Aria 90 HP
Round 2: Brom 70 HP | Aria 80 HP
Round 3: Brom 55 HP | Aria 70 HP
Round 4: Brom 50 HP | Aria 60 HP
Round 5: Brom 45 HP | Aria 50 HP
Round 6: Brom 30 HP | Aria 40 HP
Round 7: Brom 25 HP | Aria 30 HP
Round 8: Brom 20 HP | Aria 20 HP
Round 9: Brom 5 HP | Aria 10 HP
Round 10: Brom 5 HP | Aria 0 HP
Winner: Brom with 5 HP!


<Character name=Brom hp=5 damage=10>

Without weapons the damage is fixed, so this battle gives the same result
every time. The mage deals double damage (20) for the first three rounds
while the mana lasts, but the warrior's armor reduces every hit by 5. When
the mana is gone, the mage only does 10 damage per round (5 after armor).

In [10]:
# With weapons: Warrior (sword, d8) vs Mage (staff, d6)
Battle(Warrior("Brom", weapon=Weapon("Sword", Dice(8))),
       Mage("Aria", weapon=Weapon("Staff", Dice(6)))).simulate_battle()

Round 1: Brom 85 HP | Aria 95 HP
Round 2: Brom 70 HP | Aria 91 HP
Round 3: Brom 55 HP | Aria 87 HP
Round 4: Brom 55 HP | Aria 81 HP
Round 5: Brom 54 HP | Aria 80 HP
Round 6: Brom 39 HP | Aria 77 HP
Round 7: Brom 39 HP | Aria 72 HP
Round 8: Brom 38 HP | Aria 67 HP
Round 9: Brom 23 HP | Aria 62 HP
Round 10: Brom 23 HP | Aria 59 HP
Round 11: Brom 23 HP | Aria 57 HP
Round 12: Brom 8 HP | Aria 54 HP
Round 13: Brom 8 HP | Aria 51 HP
Round 14: Brom 8 HP | Aria 46 HP
Round 15: Brom 0 HP | Aria 44 HP
Winner: Aria with 44 HP!


<Character name=Aria hp=44 damage=10>

With weapons the damage is random, so the winner can change between runs.
The warrior's armor reduces each hit, while the mage's spells ignore the
weapon. This shows how dice add variation to the battle.

In [11]:
# Warrior vs Warrior
Battle(Warrior("Brom", weapon=Weapon("Axe", Dice(10))),
       Warrior("Thor", weapon=Weapon("Hammer", Dice(8)))).simulate_battle()

Round 1: Brom 100 HP | Thor 95 HP
Round 2: Brom 98 HP | Thor 94 HP
Round 3: Brom 98 HP | Thor 89 HP
Round 4: Brom 97 HP | Thor 89 HP
Round 5: Brom 96 HP | Thor 89 HP
Round 6: Brom 93 HP | Thor 89 HP
Round 7: Brom 90 HP | Thor 85 HP
Round 8: Brom 87 HP | Thor 83 HP
Round 9: Brom 87 HP | Thor 83 HP
Round 10: Brom 87 HP | Thor 80 HP
Round 11: Brom 87 HP | Thor 80 HP
Round 12: Brom 84 HP | Thor 80 HP
Round 13: Brom 84 HP | Thor 80 HP
Round 14: Brom 83 HP | Thor 80 HP
Round 15: Brom 83 HP | Thor 77 HP
Round 16: Brom 83 HP | Thor 77 HP
Round 17: Brom 83 HP | Thor 77 HP
Round 18: Brom 83 HP | Thor 73 HP
Round 19: Brom 83 HP | Thor 73 HP
Round 20: Brom 83 HP | Thor 73 HP
Round 21: Brom 80 HP | Thor 70 HP
Round 22: Brom 80 HP | Thor 70 HP
Round 23: Brom 77 HP | Thor 70 HP
Round 24: Brom 77 HP | Thor 67 HP
Round 25: Brom 77 HP | Thor 64 HP
Round 26: Brom 76 HP | Thor 61 HP
Round 27: Brom 76 HP | Thor 61 HP
Round 28: Brom 75 HP | Thor 61 HP
Round 29: Brom 75 HP | Thor 57 HP
Round 30: Brom 73 HP |

<Character name=Brom hp=51 damage=10>

Both characters have armor, so each hit is reduced by 5. Low dice rolls can
be fully blocked (damage 0), which makes the fight longer. The `max_rounds`
limit in `Battle` prevents an endless battle.

In [12]:
# Mage vs Mage
Battle(Mage("Aria"), Mage("Merlin")).simulate_battle()

Round 1: Aria 80 HP | Merlin 80 HP
Round 2: Aria 60 HP | Merlin 60 HP
Round 3: Aria 40 HP | Merlin 40 HP
Round 4: Aria 30 HP | Merlin 30 HP
Round 5: Aria 20 HP | Merlin 20 HP
Round 6: Aria 20 HP | Merlin 0 HP
Winner: Aria with 20 HP!


<Character name=Aria hp=20 damage=10>

Both mages start with 30 mana, so they cast three double-damage spells each
and then switch to base attacks. Since the hero attacks first, the hero has
a small advantage.